# 02 · Silver transform

Turns raw bronze rows into clean, typed, validated silver tables.

For each entity, this notebook works only on bronze rows that arrived since its last run (a watermark on `_ingested_at`). It then:

1. Trims text and turns blank strings into nulls.
2. Keeps the latest version of each business key.
3. Casts to proper types. A value that can't be cast fails a check instead of quietly becoming null.
4. Runs data-quality rules and referential checks (for example, every order line must point to a known product and order).
5. Sends failing rows to `dq_quarantine` with the rules they broke. Passing rows are upserted into `silver_<entity>`.

A summary of every run goes to `dq_results`.

In [ ]:
%run 00_utils

## Rules per entity

Entities run in this order so that reference tables (products, customers, orders) are current before the rows that point to them.

In [ ]:
SPECS = {
    "products": {
        "keys": ["item_no"],
        "types": {"case_price": "decimal(10,2)", "unit_cost": "decimal(10,2)", "cases_per_pallet": "int"},
        "rules": {
            "case_price_positive": "case_price > 0",
            "unit_cost_within_price": "unit_cost >= 0 AND unit_cost <= case_price",
        },
    },
    "customers": {
        "keys": ["customer_no"],
        "types": {"payment_terms_days": "int"},
        "rules": {
            "channel_known": "channel IN ('Grocery', 'Club', 'Foodservice', 'Distributor')",
            "region_known": "region IN ('North', 'South', 'East', 'West', 'Central')",
        },
    },
    "sales_order_headers": {
        "keys": ["order_no"],
        "types": {"order_no": "bigint", "order_date": "date", "requested_ship_date": "date"},
        "rules": {
            "ship_on_or_after_order": "requested_ship_date >= order_date",
            "status_known": "status IN ('Open', 'Shipped')",
        },
        "references": [("customer_no", "silver_customers", "customer_no")],
    },
    "sales_order_lines": {
        "keys": ["order_no", "line_no"],
        "types": {
            "order_no": "bigint", "line_no": "int", "cases": "int",
            "unit_price": "decimal(10,2)", "discount_pct": "decimal(5,4)", "net_amount": "decimal(12,2)",
        },
        "rules": {
            "cases_positive": "cases > 0",
            "discount_in_range": "discount_pct BETWEEN 0 AND 0.5",
            "net_amount_matches": "abs(net_amount - round(cases * unit_price * (1 - discount_pct), 2)) <= 0.01",
        },
        "references": [
            ("order_no", "silver_sales_order_headers", "order_no"),
            ("item_no", "silver_products", "item_no"),
        ],
    },
    "inventory_snapshots": {
        "keys": ["snapshot_date", "item_no"],
        "types": {"snapshot_date": "date", "cases_on_hand": "int", "cases_on_hold_qa": "int"},
        "rules": {
            "on_hand_not_negative": "cases_on_hand >= 0",
            "hold_within_on_hand": "cases_on_hold_qa BETWEEN 0 AND cases_on_hand",
        },
        "references": [("item_no", "silver_products", "item_no")],
    },
    "production_runs": {
        "keys": ["run_id"],
        "types": {
            "run_id": "bigint", "production_date": "date", "planned_minutes": "int", "downtime_minutes": "int",
            "ideal_cases_per_minute": "double", "total_cases": "int", "good_cases": "int",
        },
        "rules": {
            "downtime_within_planned": "downtime_minutes BETWEEN 0 AND planned_minutes",
            "good_within_total": "good_cases BETWEEN 0 AND total_cases",
            "ideal_rate_positive": "ideal_cases_per_minute > 0",
        },
        "references": [("item_no", "silver_products", "item_no")],
    },
    "downtime_events": {
        # The source has no id for an event, so the key is a hash of the whole row.
        "keys": ["event_id"],
        "hash_key": "event_id",
        "types": {"run_id": "bigint", "production_date": "date", "minutes": "int"},
        "rules": {
            "minutes_positive": "minutes > 0",
            "category_known": "category IN ('Planned', 'Unplanned')",
        },
        "references": [("run_id", "silver_production_runs", "run_id")],
    },
    "weather_daily": {
        "keys": ["region", "weather_date"],
        "types": {"weather_date": "date"},
        "rules": {
            "max_at_least_min": "temperature_2m_max >= temperature_2m_min",
            "precipitation_not_negative": "precipitation_sum >= 0",
        },
    },
}

LINEAGE = ["_source_file", "_batch_id", "_ingested_at"]
WATERMARKS = "etl_silver_watermark"
BATCH_ID = new_batch_id()
RUN_AT = datetime.now(timezone.utc)

## Transform

In [ ]:
def get_watermark(entity):
    if not table_exists(WATERMARKS):
        return None
    row = spark.table(WATERMARKS).where(F.col("entity") == entity).select("last_ingested_at").first()
    return row.last_ingested_at if row else None


def set_watermark(entity, value):
    schema = "entity string, last_ingested_at timestamp, updated_at timestamp"
    upsert(spark.createDataFrame([(entity, value, RUN_AT)], schema), WATERMARKS, ["entity"])


def transform(entity, spec):
    bronze, silver = f"bronze_{entity}", f"silver_{entity}"
    if not table_exists(bronze):
        log(f"  {entity:<22} no bronze table yet")
        return None

    source = spark.table(bronze)
    watermark = get_watermark(entity)
    if watermark is not None:
        source = source.where(F.col("_ingested_at") > F.lit(watermark))
    new_watermark = source.agg(F.max("_ingested_at")).first()[0]
    if new_watermark is None:
        log(f"  {entity:<22} nothing new")
        return None

    keys, types = spec["keys"], spec["types"]
    data_cols = [c for c in source.columns if not c.startswith("_")]

    # 1. Trim text and turn blanks into nulls.
    def clean(c):
        if dict(source.dtypes)[c] != "string":
            return F.col(c)
        trimmed = F.trim(F.col(c))
        return F.when(trimmed == "", None).otherwise(trimmed).alias(c)

    df = source.select(*[clean(c) for c in data_cols], *LINEAGE)
    if spec.get("hash_key"):
        df = df.withColumn(spec["hash_key"], F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in data_cols]), 256))

    # 2. Latest version of each key wins.
    latest = Window.partitionBy(*keys).orderBy(F.col("_ingested_at").desc(), F.col("_source_file").desc())
    df = df.withColumn("_rn", F.row_number().over(latest)).where("_rn = 1").drop("_rn")

    # 3. Cast, keeping the raw text so a failed cast can be reported.
    checks = {"key_present": " AND ".join(f"`{k}` IS NOT NULL" for k in keys)}
    for col, dtype in types.items():
        df = df.withColumn(f"_raw_{col}", F.col(col)).withColumn(col, F.expr(f"try_cast(`{col}` AS {dtype})"))
        checks[f"{col}_is_{dtype.split('(')[0]}"] = f"`_raw_{col}` IS NULL OR `{col}` IS NOT NULL"

    # 4. Business rules, then referential checks against silver tables.
    checks.update(spec["rules"])
    for col, ref_table, ref_col in spec.get("references", []):
        flag = f"_has_{col}_in_{ref_table}"
        if table_exists(ref_table):
            ref = spark.table(ref_table).select(F.col(ref_col).alias("_ref")).distinct()
            df = df.join(ref, df[col] == ref["_ref"], "left").withColumn(flag, F.col("_ref").isNotNull()).drop("_ref")
        else:
            df = df.withColumn(flag, F.lit(False))
        checks[f"{col}_exists_in_{ref_table}"] = flag

    failures = [F.when(~F.coalesce(F.expr(cond), F.lit(False)), F.lit(name)) for name, cond in checks.items()]
    df = df.withColumn("_failed", F.filter(F.array(*failures), lambda x: x.isNotNull())).cache()

    # 5. Split: failing rows to quarantine, the rest to silver.
    output_cols = data_cols + ([spec["hash_key"]] if spec.get("hash_key") else [])
    raw_record = F.to_json(F.struct(*[F.col(f"_raw_{c}").alias(c) if c in types else F.col(c) for c in data_cols]))

    bad = df.where(F.size("_failed") > 0)
    good = df.where(F.size("_failed") == 0).select(*output_cols, *LINEAGE)

    rows_in, rows_bad = df.count(), bad.count()
    if rows_bad:
        append(
            bad.select(
                F.lit(entity).alias("entity"),
                F.array_join("_failed", ", ").alias("failed_checks"),
                raw_record.alias("record"),
                *LINEAGE,
                F.lit(BATCH_ID).alias("quarantine_batch_id"),
                F.lit(RUN_AT).alias("quarantined_at"),
            ),
            "dq_quarantine",
        )
    upsert(good, silver, keys)
    set_watermark(entity, new_watermark)
    df.unpersist()

    log(f"  {entity:<22} {rows_in:>8,} in  {rows_in - rows_bad:>8,} to silver  {rows_bad:>5,} quarantined")
    return (entity, BATCH_ID, rows_in, rows_in - rows_bad, rows_bad, RUN_AT)


log(f"Batch {BATCH_ID}")
results = [r for r in (transform(e, s) for e, s in SPECS.items()) if r]

if results:
    schema = "entity string, batch_id string, rows_in long, rows_to_silver long, rows_quarantined long, run_at timestamp"
    append(spark.createDataFrame(results, schema), "dq_results")